# Machine Learning Model

This notebook builds machine learning models to classify walking and running activities using accelerometer and gyroscope sensor data.

In [ ]:
import pandas as pd
import psycopg2
import numpy as np

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    database="iot_motion_project",
    user="postgres",
    password="password"
)

In [ ]:
query = "SELECT * FROM motion_data"

df = pd.read_sql(query, conn)

df.head()

In [ ]:
df['magnitude'] = (
    (df['acc_x']**2 +
     df['acc_y']**2 +
     df['acc_z']**2) ** 0.5
)

## Feature Selection

This step selects the input features (X) and target variable (y) for machine learning classification.

In [ ]:
X = df[['acc_x', 'acc_y', 'acc_z',
        'gyro_x', 'gyro_y', 'gyro_z',
        'magnitude']]

y = df['activity']

X.head()

## Train-Test Split

This step splits the dataset into training and testing sets to evaluate model performance on unseen data.

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## Logistic Regression Model

This model predicts walking and running activities using sensor features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_model = LogisticRegression()

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Logistic Regression Accuracy:", accuracy)

## Confusion Matrix

This shows how well the model distinguishes walking and running activities.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

## Random Forest Model

This model improves classification performance and helps identify the most important features for activity prediction.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

## Feature Importance

This analysis identifies which sensor features contribute most to activity classification.

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance

### Feature Importance Visualization

This chart shows which sensor features contribute most to activity classification in the Random Forest model.

`acc_y` is the most important feature, confirming the correlation analysis from the EDA stage and showing that acceleration in the y-axis is the strongest indicator for distinguishing walking and running activities.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance Score")

plt.savefig("Desktop/iot_motion/iot_motion_images/feature_importance.png")

plt.show()

In [ ]:
df['magnitude'] = (
    (df['acc_x']**2 +
     df['acc_y']**2 +
     df['acc_z']**2) ** 0.5
)

df.to_csv("Desktop/iot_motion/final_motion_data.csv", index=False)